# Big Data Lab Work

In [1]:
# %pip install --force-reinstall "ipykernel==6.28.0"

In [2]:
import ipykernel
print(ipykernel.__version__)

6.28.0


In [3]:
!ls -lh /mnt/data/public/polymarket/

total 568K
-rwxrwxr-x+ 1 root root 9.5K Mar 27 16:52 README.md
-rwxrwxr-x+ 1 root root 523K Mar 27 16:52 banner.png
drwxrwxr-x+ 2 root root 4.0K Apr 26 11:42 features
drwxrwxr-x+ 2 root root 4.0K Apr 26 11:42 labels
drwxrwxr-x+ 2 root root 4.0K Apr 26 11:42 orderbook
drwxrwxr-x+ 2 root root 4.0K Apr 26 11:42 snapshots


In [4]:
!nvidia-smi

No devices were found


In [5]:
!ls -lh /mnt/data/public/polymarket/features

total 31M
-rwxrwxr-x+ 1 root root 31M Mar 27 16:52 ml_features_1m_v2.parquet


In [6]:
ls -lh /mnt/data/public/polymarket/labels

total 48M
-rwxrwxr-x+ 1 root root 15M Mar 27 16:52 market_targets.parquet*
-rwxrwxr-x+ 1 root root 33M Mar 27 16:52 trades.parquet*


In [7]:
ls -lh /mnt/data/public/polymarket/orderbook | head -5

total 35G
-rwxrwxr-x+ 1 root root 2.0G Mar 27 16:53 orderbook_2026-03-06.parquet*
-rwxrwxr-x+ 1 root root 2.1G Mar 27 16:55 orderbook_2026-03-07.parquet*
-rwxrwxr-x+ 1 root root 2.2G Mar 27 16:58 orderbook_2026-03-08.parquet*
-rwxrwxr-x+ 1 root root 1.3G Mar 27 17:00 orderbook_2026-03-09.parquet*


In [8]:
ls -lh /mnt/data/public/polymarket/snapshots | head -5

total 3.9G
-rwxrwxr-x+ 1 root root 302M Mar 27 17:32 snapshots_2026-03-06.parquet*
-rwxrwxr-x+ 1 root root 224M Mar 27 17:32 snapshots_2026-03-07.parquet*
-rwxrwxr-x+ 1 root root 228M Mar 27 17:32 snapshots_2026-03-08.parquet*
-rwxrwxr-x+ 1 root root  81M Mar 27 17:33 snapshots_2026-03-09.parquet*


In [9]:
!du -sch /mnt/data/public/polymarket/*

12K	/mnt/data/public/polymarket/README.md
524K	/mnt/data/public/polymarket/banner.png
31M	/mnt/data/public/polymarket/features
48M	/mnt/data/public/polymarket/labels
35G	/mnt/data/public/polymarket/orderbook
3.9G	/mnt/data/public/polymarket/snapshots
39G	total


## Reading OrderBook

In [10]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("View Polymarket Orderbook Parquet")
    .getOrCreate()
)

path = "/mnt/data/public/polymarket/orderbook/orderbook_2026-03-06.parquet"

df = spark.read.parquet(path)

# Show schema
df.printSchema()

# Show first 20 rows without truncating columns
df.show(20, truncate=False)

# Optional: see column names
print(df.columns)

# Optional: count rows, can be slow for 2GB+
print("Row count:", df.count())

root
 |-- timestamp_received: long (nullable = true)
 |-- timestamp_created_at: long (nullable = true)
 |-- market_id: string (nullable = true)
 |-- best_bid: float (nullable = true)
 |-- best_ask: float (nullable = true)
 |-- change_price: float (nullable = true)
 |-- change_size: float (nullable = true)
 |-- change_side: string (nullable = true)
 |-- token_id: string (nullable = true)
 |-- spread: float (nullable = true)
 |-- mid_price: float (nullable = true)

+------------------+--------------------+------------------------------------------------------------------+--------+--------+------------+-----------+-----------+-----------------------------------------------------------------------------+------------+---------+
|timestamp_received|timestamp_created_at|market_id                                                         |best_bid|best_ask|change_price|change_size|change_side|token_id                                                                     |spread      |mid_price|
+-

In [11]:
df.limit(10).toPandas()

,timestamp_received,timestamp_created_at,market_id,best_bid,best_ask,change_price,change_size,change_side,token_id,spread,mid_price
0,1772755441639,1772755441679,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.009,1000.900024,BUY,4455468110807479331389362642427847115009165823...,0.004,0.011
1,1772755573852,1772755578267,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.133,0.000000,SELL,4455468110807479331389362642427847115009165823...,0.004,0.011
2,1772755574100,1772755578472,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.133,40.000000,SELL,4455468110807479331389362642427847115009165823...,0.004,0.011
3,1772755577386,1772755579670,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.043,256.700012,SELL,4455468110807479331389362642427847115009165823...,0.004,0.011
4,1772755658205,1772755658227,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.009,900.900024,BUY,4455468110807479331389362642427847115009165823...,0.004,0.011
5,1772755964391,1772755964470,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.154,0.000000,SELL,4455468110807479331389362642427847115009165823...,0.004,0.011
6,1772756105891,1772756105922,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.009,1000.900024,BUY,4455468110807479331389362642427847115009165823...,0.004,0.011
7,1772756146031,1772756146049,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.699,200.000000,SELL,4455468110807479331389362642427847115009165823...,0.004,0.011
8,1772756200595,1772756200608,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.043,250.000000,SELL,4455468110807479331389362642427847115009165823...,0.004,0.011
9,1772756240758,1772756240799,0x00000977017fa72fb6b1908ae694000d3b51f442c255...,0.009,0.013,0.699,0.000000,SELL,4455468110807479331389362642427847115009165823...,0.004,0.011


In [ ]:
# df.groupBy("update_type") \
#   .count() \
#   .orderBy(col("count").desc()) \
#   .show(truncate=False)

## Reading Snapshots

In [13]:
path = "/mnt/data/public/polymarket/snapshots/snapshots_2026-03-06.parquet"

df2 = spark.read.parquet(path)
df2.limit(10).toPandas()

,timestamp_received,timestamp_created_at,market_id,update_type,data
0,1772755479569,1772755479636,0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e2...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
1,1772757306842,1772757306884,0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e2...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
2,1772758741660,1772758741701,0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e2...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
3,1772758749082,1772758749140,0x0007deb167d0bb816e2e847a15435f3e384f97f9e3e2...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
4,1772757678461,1772757678492,0x0008043c3ed513ecff7ee64380fc943dc73eb3dfb667...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
5,1772757825635,1772757825916,0x0008043c3ed513ecff7ee64380fc943dc73eb3dfb667...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
6,1772758000770,1772758000792,0x0008043c3ed513ecff7ee64380fc943dc73eb3dfb667...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
7,1772758045967,1772758045980,0x0008043c3ed513ecff7ee64380fc943dc73eb3dfb667...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
8,1772758055224,1772758055235,0x0008043c3ed513ecff7ee64380fc943dc73eb3dfb667...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."
9,1772756288585,1772756288645,0x004230fb1f54a139d50ba2041e062a01461c931a6725...,book_snapshot,"{""update_type"": ""book_snapshot"", ""market_id"": ..."


## Reading Labels

In [14]:
# path = "/mnt/data/public/polymarket/snapshots/snapshots_2026-03-06.parquet"
path = "/mnt/data/public/polymarket/labels/market_targets.parquet"

df3 = spark.read.parquet(path)
df3.limit(100).toPandas()

,condition_id,question,category,end_date,closed,uma_status,volume,liquidity,clob_token_id_yes,clob_token_id_no,target
0,0xb48621f7eba07b0a3eeabc6afb09ae42490239903997...,BitBoy convicted?,,2026-03-31T12:00:00Z,False,,2.266330e+05,9631.43077,7546712961590831958303147464265888547913563043...,3842963720267267286970642336860752782302644680...,NaN
1,0x9c1a953fe92c8357f1b646ba25d983aa83e90c525992...,Russia-Ukraine Ceasefire before GTA VI?,,2026-07-31T12:00:00Z,False,,1.403228e+06,100309.56630,8501497159083948713316135768103773293754490207...,2527312495175492857904889758552137141356236738...,NaN
2,0x1fad72fae204143ff1c3035e99e7c0f65ea8d5cd9bd1...,New Rihanna Album before GTA VI?,,2026-07-31T12:00:00Z,False,,6.751036e+05,32729.52530,9802249026969240999812649612759703249033407008...,5383155306188300653073987728410593891972140877...,NaN
3,0x50ddb9cd80d5c271664a2ebb7fcaed1d0a148d82c8e8...,New Playboi Carti Album before GTA VI?,,2026-07-31T12:00:00Z,False,,7.094616e+05,29544.93060,8827504006008477337655718797221526751304984864...,9437620581602295554297963554227993296735991576...,NaN
4,0x32b09f6390252b37d674501527e709016d55581b2c1e...,Will Jesus Christ return before GTA VI?,,2026-07-31T12:00:00Z,False,,1.027764e+07,831771.48120,9043581125366557801495738082650599253005407769...,9238862908268180562280162270352898292254328635...,NaN
...,...,...,...,...,...,...,...,...,...,...,...
95,0x7876851632c295043c66536150a304cb785abdf712ba...,Will Uruguay win the 2026 FIFA World Cup?,,2026-07-20T00:00:00Z,False,,6.494237e+06,187207.19916,9723912606267331024376361723664439294553035614...,1929169204037852961891791059972757124230593502...,NaN
96,0x5ccfe1b69a582d2985db08a8481a0d74c314b1fce9b4...,Will Mexico win the 2026 FIFA World Cup?,,2026-07-20T00:00:00Z,False,,6.445826e+06,378449.06121,2258777530186914674823791305050593248564895848...,8904100647536478935880502613965067780708769898...,NaN
97,0x32cfa52198e85e070d1b17d1b53c5c3a6aaae7736cdc...,Will Belgium win the 2026 FIFA World Cup?,,2026-07-20T00:00:00Z,False,,6.910787e+06,386578.77418,3081580706745663152451053500261710620541783289...,7114588899488815329244262301975051762253540747...,NaN
98,0xe99cc59f32b10d23acf196d1a0e8264ea30fca198428...,Will Colombia win the 2026 FIFA World Cup?,,2026-07-20T00:00:00Z,False,,6.776060e+06,592672.33030,9880339017552145671265367828047492063793459623...,6682696535116667515588751516730608630741233222...,NaN


# Big Data Mining

## Loading Data

In [15]:
# %pip install polars

In [16]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
import re
from datetime import datetime

warnings.filterwarnings("ignore")

DATA_DIR = "/mnt/data/public/polymarket"
ob_dir = f"{DATA_DIR}/orderbook"
snap_dir = f"{DATA_DIR}/snapshots"

# -----------------------------
# Date filter config
# -----------------------------
# Option A: range (inclusive)
START_DATE = "2026-03-06"   # set None to disable
END_DATE   = "2026-03-06"   # set None to disable

# Option B: explicit dates (inclusive set)
# If provided, this takes priority over range.
INCLUDE_DATES = None
# Example:
# INCLUDE_DATES = {"2026-03-01", "2026-03-03", "2026-03-06"}

date_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})")

def extract_date(fname):
    m = date_pattern.search(fname)
    return m.group(1) if m else None

def keep_file_by_date(fname):
    d = extract_date(fname)
    if d is None:
        return False

    if INCLUDE_DATES is not None:
        return d in INCLUDE_DATES

    if START_DATE is None and END_DATE is None:
        return True

    dt = datetime.strptime(d, "%Y-%m-%d").date()
    if START_DATE is not None:
        start = datetime.strptime(START_DATE, "%Y-%m-%d").date()
        if dt < start:
            return False
    if END_DATE is not None:
        end = datetime.strptime(END_DATE, "%Y-%m-%d").date()
        if dt > end:
            return False
    return True

ob_files_all = sorted([f for f in os.listdir(ob_dir) if f.endswith(".parquet")])
snap_files_all = sorted([f for f in os.listdir(snap_dir) if f.endswith(".parquet")])

ob_files = [f for f in ob_files_all if keep_file_by_date(f)]
snap_files = [f for f in snap_files_all if keep_file_by_date(f)]

print(f"Orderbook files selected: {len(ob_files)}")
print(f"Snapshot files selected:  {len(snap_files)}")

for f in ob_files:
    size_mb = os.path.getsize(f"{ob_dir}/{f}") / (1024**2)
    print(f"  {f}  ({size_mb:,.0f} MB)")


: 

: 

: 

In [ ]:
# Load a sample — evenly spaced across the full day (safe: ~200 MB RAM)
# We use the second-to-last file (a full 24-hour day) for better EDA coverage
# Using every 500th row gives us coverage across all 24 hours
sample_file = ob_files[-2] if len(ob_files) > 1 else ob_files[0]
sample = (
    pl.scan_parquet(f"{DATA_DIR}/orderbook/{sample_file}")
    .with_row_index("row_idx")
    .filter(pl.col("row_idx") % 500 == 0)
    .drop("row_idx")
    .collect()
)

print(f"Sample: {sample.height:,} rows, {sample.width} columns")
print(f"Columns: {sample.columns}")
print(f"\nUnique markets in sample: {sample['market_id'].n_unique():,}")
print(f"Time range: {sample['timestamp_received'].min()} → {sample['timestamp_received'].max()}")
print()
print(sample.head(5))

NameError: name 'ob_files' is not defined

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# spark = (
#     SparkSession.builder
#     .appName("Polymarket Sample Every 500th Row")
#     .getOrCreate()
# )

# same path style you use
path = f"{DATA_DIR}/orderbook/{sample_file}"

df = spark.read.parquet(path)

# Fast approximate: ~1 out of 500 rows
sample_spark = df.sample(withReplacement=False, fraction=1/500, seed=42)

sample_spark.show(5, truncate=False)

NameError: name 'DATA_DIR' is not defined